# FIT · Step 3: Baseline RAPM

Before building FIT, I need the baseline it has to beat: plain **RAPM** (Regularized Adjusted Plus-Minus).

- **One row per stint per team on offense.** The 5 players on offense get a 1 in their offense column, and the 5 defenders get a 1 in their defense column.
- **Target:** points scored per 100 possessions in that stint.
- **Weight:** possessions, so a 20-possession stint counts more than a 2-possession one.
- **Ridge regression:** the penalty (alpha) pulls everyone toward average unless the data clearly says otherwise. Alpha is picked with cross-validation, splitting **by game** so stints from the same game never end up in both training and testing.

Each player's offense coefficient is his O-RAPM. His defense coefficient is flipped (allowing fewer points is good), giving D-RAPM. RAPM = O-RAPM + D-RAPM, in points per 100 possessions.

In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse

from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

from matplotlib import pyplot as plt
import seaborn as sns; sns.set_style('whitegrid')

pd.set_option('display.max_columns', 100)

In [ ]:
SEASON = '2025-26'

stints = pd.read_csv('data/stints.csv', dtype = {'GAME_ID': str})
players = pd.read_csv('data/players.csv')

print(f'Stints: {len(stints)}, Games: {stints["GAME_ID"].nunique()}')

Leave out garbage time (down 20+ in the last 5 minutes), since starters are usually sitting and nobody is really trying.

In [ ]:
print(f'Garbage time stints removed: {stints["GARBAGE"].sum()}')

stints = stints.loc[~stints['GARBAGE']]

## Build the matrix

Each stint becomes two rows: the home team on offense and the away team on offense.

In [ ]:
home_cols = ['H1', 'H2', 'H3', 'H4', 'H5']
away_cols = ['A1', 'A2', 'A3', 'A4', 'A5']

off_ids = np.vstack([stints[home_cols].values, stints[away_cols].values])
def_ids = np.vstack([stints[away_cols].values, stints[home_cols].values])

pts = np.concatenate([stints['PTS_HOME'].values, stints['PTS_AWAY'].values])
poss = np.concatenate([stints['POSS_HOME'].values, stints['POSS_AWAY'].values])
is_home = np.concatenate([np.ones(len(stints)), np.zeros(len(stints))])
game_ids = np.concatenate([stints['GAME_ID'].values, stints['GAME_ID'].values])

keep = poss > 0

off_ids, def_ids, pts, poss, is_home, game_ids = off_ids[keep], def_ids[keep], pts[keep], poss[keep], is_home[keep], game_ids[keep]

y = 100 * pts / poss
w = poss

print(f'Rows: {len(y)}')
print(f'Average points per 100 possessions: {np.average(y, weights = w):.1f}')

In [ ]:
player_ids = np.unique(np.concatenate([off_ids.ravel(), def_ids.ravel()]))
n = len(player_ids)

col = pd.Series(np.arange(n), index = player_ids)

rows = np.repeat(np.arange(len(y)), 5)
off_cols = col.loc[off_ids.ravel()].values
def_cols = col.loc[def_ids.ravel()].values + n

X = sparse.csr_matrix((np.ones(2 * len(rows)), (np.concatenate([rows, rows]), np.concatenate([off_cols, def_cols]))),
                      shape = (len(y), 2 * n))

X = sparse.hstack([X, sparse.csr_matrix(is_home.reshape(-1, 1))]).tocsr()

print(f'Players: {n}')
print(f'Matrix shape: {X.shape}  (offense columns + defense columns + home court)')

## Pick alpha with cross-validation

Too small and the ratings are mostly noise. Too big and everyone gets pulled to average. Each alpha is scored on games the model didn't see.

In [ ]:
alphas = [300, 1000, 3000, 10000, 30000]
gkf = GroupKFold(n_splits = 5)

cv_results = []

for alpha in alphas:

    errors = []

    for train_idx, test_idx in gkf.split(X, y, groups = game_ids):

        model = Ridge(alpha = alpha)
        model.fit(X[train_idx], y[train_idx], sample_weight = w[train_idx])

        pred = model.predict(X[test_idx])
        errors.append(np.average((y[test_idx] - pred) ** 2, weights = w[test_idx]))

    cv_results.append({'alpha': alpha, 'error': np.mean(errors)})
    print(f'alpha = {alpha}: error = {np.mean(errors):.1f}')

cv_results = pd.DataFrame(cv_results)
best_alpha = cv_results.sort_values('error')['alpha'].iloc[0]

print(f'Best alpha: {best_alpha}')

In [ ]:
model = Ridge(alpha = best_alpha)
model.fit(X, y, sample_weight = w)

poss_played = np.bincount(off_cols, weights = np.repeat(w, 5), minlength = 2 * n)[:n] + \
              np.bincount(def_cols - n, weights = np.repeat(w, 5), minlength = n)

rapm = pd.DataFrame({'PERSON_ID': player_ids,
                     'O_RAPM': model.coef_[:n],
                     'D_RAPM': -model.coef_[n:2 * n],
                     'POSS': poss_played})

rapm['RAPM'] = rapm['O_RAPM'] + rapm['D_RAPM']
rapm = rapm.merge(players[['PERSON_ID', 'PLAYER']], on = 'PERSON_ID', how = 'left')

rapm = rapm[['PLAYER', 'PERSON_ID', 'POSS', 'O_RAPM', 'D_RAPM', 'RAPM']]

print(f'Home court advantage: {model.coef_[-1]:+.2f} points per 100 possessions')

rapm.to_csv(f'data/rapm_{SEASON}.csv', index = False)

## Results

Players need enough possessions to be listed (both ends of the floor count). With a full season, 2,000 is about 1,000 minutes.

In [ ]:
MIN_POSS = 2000 if stints['GAME_ID'].nunique() > 500 else 100

top = rapm.loc[rapm['POSS'] >= MIN_POSS].sort_values('RAPM', ascending = False)

top.head(15).round(2)

In [ ]:
top.tail(10).round(2)

In [ ]:
rapm.loc[rapm['PLAYER'].str.contains('Gilgeous-Alexander|Jok|Wembanyama', na = False)].round(2)

In [ ]:
plt.figure(figsize = (8, 7))

best = top.head(15).sort_values('RAPM')

plt.barh(best['PLAYER'], best['RAPM'], color = '#1D9E75')

plt.xlabel('Points per 100 possessions vs. average')
plt.title(f'Baseline RAPM, top 15 ({SEASON})')
plt.show()

## What this baseline can't do (and why FIT exists)

- **One season of RAPM is noisy.** Some names in the top 15 will look strange. That's the small-sample problem, not a bug.
- **It uses actual points.** If opponents hit a lot of open threes, the defenders get blamed. FIT Layer 1 switches to expected points.
- **It assumes a player adds the same amount in every lineup.** That's fit, and it's the main thing FIT is built to measure.

Every FIT layer has to beat this baseline at predicting games it hasn't seen. If it doesn't, it doesn't make it in.